[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/07_traing_problem_and%20soultion/02_loss_landscape_optimization/02_loss_landscape_optimization.ipynb)

# 02. Loss Landscape, Learning Rate & Optimizers

---


In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os, sys

if 'google.colab' in sys.modules:
    !git clone https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git
    os.chdir('Multimodal-Deep-Learning')
    os.chdir('07_traing_problem_and soultion/02_loss_landscape_optimization')
    !pip install -q torch torchvision matplotlib numpy
else:
    nb_dir = os.getcwd()
    if not os.path.basename(nb_dir) == '02_loss_landscape_optimization':
        os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..', '02_loss_landscape_optimization'))
    sys.path.append(os.path.join(os.getcwd(), '..', '..'))

print(f'Working directory: {os.getcwd()}')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11})


## 1. Loss Landscape — Why Training Diverges

High-dimensional loss surfaces have saddle points (not local minima). **Sharp minima** generalize poorly; **flat minima** generalize better.

```
        Loss
         |    \ sharp minimum (bad gen.)
         |     \___/
         |  ___/   \___  flat minimum (good gen.)
         |_/             \___
         +--------------------> weight
```


## 2. Learning Rate Too High — Quadratic Proof

For $\mathcal{L}(w) = \frac{\lambda}{2}w^2$, GD update $w_{t+1} = (1 - \eta\lambda) w_t$.

**Divergence when** $\lvert 1 - \eta\lambda \rvert > 1$ → $\eta > 2/\lambda$.

**Oscillation when** $\eta\lambda > 1$ (sign flips each step).


In [ ]:
lam = 1.0
w = 1.0
for eta, label in [(0.5, 'stable'), (1.5, 'oscillate'), (3.0, 'diverge')]:
    ws = [1.0]
    for _ in range(20):
        ws.append(ws[-1] * (1 - eta * lam))
    plt.plot(ws, label=f'eta={eta} ({label})')
plt.axhline(0, color='k', lw=0.5); plt.legend(); plt.title('GD on L=0.5*lam*w^2, lam=1'); plt.show()


## 3. SGD + Momentum

$$
v_t = \beta v_{t-1} + g_t, \quad w_t = w_{t-1} - \eta v_t
$$

Effective learning rate increases in consistent gradient directions; dampens oscillations in ravines.


## 4. Adam — Bias Correction Proof

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2
$$

$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

**Why:** $E[m_t] = (1-\beta_1^t) E[g_t]$ at initialization → dividing removes bias toward zero.


In [ ]:
def adam_bias_correction(beta1=0.9, steps=10):
    m, g = 0.0, 1.0  # constant gradient
    raw, corrected = [], []
    for t in range(1, steps+1):
        m = beta1 * m + (1 - beta1) * g
        raw.append(m)
        corrected.append(m / (1 - beta1**t))
    return raw, corrected

raw, corr = adam_bias_correction()
plt.plot(raw, 'o-', label='Raw m_t'); plt.plot(corr, 's-', label='Bias-corrected m_hat')
plt.axhline(1.0, ls='--', color='gray', label='True E[g]'); plt.legend(); plt.title('Adam bias correction'); plt.show()


## 5. AdamW vs Adam — Weight Decay $\neq$ L2

Adam + L2 adds $\lambda w$ to the **gradient** (coupled with adaptive rates). AdamW decouples:

$$
w_t \leftarrow w_{t-1} - \eta\left(\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon} + \lambda w_{t-1}\right)
$$

Loshchilov & Hutter show AdamW generalizes better than Adam + L2.


## 6. LR Schedulers

**Cosine annealing:** $\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})(1 + \cos(\pi t / T))$

**Warmup + cosine:** linear ramp 0 → $\eta_{max}$ for $W$ steps, then cosine decay.


In [ ]:
T, W, eta_max, eta_min = 100, 10, 1e-2, 1e-5
lrs = []
for t in range(T):
    if t < W:
        lr = eta_max * t / W
    else:
        prog = (t - W) / (T - W)
        lr = eta_min + 0.5 * (eta_max - eta_min) * (1 + np.cos(np.pi * prog))
    lrs.append(lr)
plt.plot(lrs); plt.xlabel('Step'); plt.ylabel('LR'); plt.title('Warmup + Cosine Annealing'); plt.show()


## 7. Compare SGD, Adam, AdamW on Same Problem


In [ ]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

def train_opt(opt_name, lr=1e-2, wd=0.01, steps=100):
    torch.manual_seed(0)
    model = TinyMLP()
    x = torch.randn(64, 10); y = torch.randn(64)
    if opt_name == 'SGD':
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    elif opt_name == 'Adam':
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    losses = []
    for _ in range(steps):
        opt.zero_grad()
        loss = F.mse_loss(model(x), y)
        loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

for name in ['SGD', 'Adam', 'AdamW']:
    plt.plot(train_opt(name), label=name)
plt.xlabel('Step'); plt.ylabel('Loss'); plt.legend(); plt.title('Optimizer comparison'); plt.yscale('log'); plt.show()


## 8. Scheduler Decision Table

| Scheduler | Formula | When to Use |
|-----------|---------|-------------|
| Step decay | $\eta \times \gamma$ every $K$ epochs | Classic CV baselines |
| Cosine | smooth decay to $\eta_{min}$ | Transformers, CLIP |
| Warmup+cosine | linear warmup then cosine | Large models, Adam |
| One-cycle | ramp up then down | Fast convergence (Smith 2017) |


## References & Further Reading

- Kingma & Ba (2015) — Adam — [arXiv:1412.6980](https://arxiv.org/abs/1412.6980)
- Loshchilov & Hutter (2019) — Decoupled Weight Decay (AdamW) — [arXiv:1711.05101](https://arxiv.org/abs/1711.05101)
- Smith (2017) — Cyclical Learning Rates — [arXiv:1506.01186](https://arxiv.org/abs/1506.01186)

**Blog posts:**
- [Lilian Weng — LR Schedules](https://lilianweng.github.io/posts/2021-12-05-large-batch/)
